In [ ]:
# 17_honest_did_extensions.ipynb
# Honest-DiD extensions: post-period AVERAGE target per group, and a DIRECT honest-DiD
# on the buffer (difference of the two disjoint CS group event studies). Writes t19.
# Honest-DiD extensions: (A) post-period AVERAGE target for each group; (B) a DIRECT
# honest-DiD on the BUFFER = CS event study of (No-PHI minus Has-PHI). The two groups
# are disjoint samples, so the buffer's covariance is the sum of the group covariances.
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, moderndid as md
from moderndid import create_sensitivity_results_rm, construct_original_cs
PROC, TAB = "../data/processed", "../results/tables"
p=pd.read_parquet(f"{PROC}/khp_panel.parquet"); p["che40f"]=p["che40"].astype(float)
dd=p[["PIDWON","year","cohort","che40f","base_phi"]].dropna(subset=["che40f"])

def cs_vec(df):
    mp=md.att_gt(data=df,yname="che40f",tname="year",idname="PIDWON",gname="cohort",
                 control_group="notyettreated",est_method="dr",base_period="universal",
                 panel=True,allow_unbalanced_panel=True,boot=False)
    dyn=md.aggte(mp,type="dynamic",na_rm=True)
    et=np.asarray(dyn.event_times); att=np.asarray(dyn.att_by_event); se=np.asarray(dyn.se_by_event)
    IF=np.asarray(dyn.influence_func); n=IF.shape[0]; V=IF.T@IF/n**2
    keep=(se>0)|(et==et[np.argmin(np.abs(et+1))])  # keep all but we will drop ref below
    return et, att, V

def order_prepost(et, att, V):
    ref = -1
    mask = et != ref
    order = np.argsort(et[mask]); idx = np.where(mask)[0][order]
    b = att[idx]; Vo = V[np.ix_(idx,idx)]; ets = et[idx]
    npre = int((ets<0).sum()); npost = int((ets>=0).sum())   # k=0 is the first POST period
    return b, Vo, npre, npost, ets

def robust(b, V, npre, npost, lvec, mbar=(0.5,1.0,1.5,2.0,3.0)):
    oc = construct_original_cs(b, V, npre, npost, l_vec=lvec)
    rc = create_sensitivity_results_rm(b, V, npre, npost, l_vec=lvec, m_bar_vec=list(mbar))
    rcp = rc.to_pandas() if hasattr(rc,"to_pandas") else pd.DataFrame(rc)
    # normalize column names
    cols={c.lower():c for c in rcp.columns}
    lbc=cols.get("lb","lb"); ubc=cols.get("ub","ub"); mbc=[c for c in rcp.columns if "bar" in c.lower() or c.lower()=="m"][0]
    out=[]; bd=None
    for _,r in rcp.iterrows():
        lb,ub,mb=float(r[lbc]),float(r[ubc]),float(r[mbc]); inc=not(lb>0 or ub<0)
        out.append((mb,lb,ub,inc)); bd=mb if (bd is None and inc) else bd
    olb=float(getattr(oc,"lb",oc[0] if hasattr(oc,'__getitem__') else np.nan))
    oub=float(getattr(oc,"ub",oc[1] if hasattr(oc,'__getitem__') else np.nan))
    return olb,oub,out,bd

# group event studies
E={}
for key,lab in [(0,"No PHI"),(1,"Has PHI")]:
    et,att,V=cs_vec(dd[dd["base_phi"]==key]); E[lab]=(et,att,V)

print("=== (A) POST-AVERAGE honest-DiD by group (CHE40) ===")
for lab in ["No PHI","Has PHI"]:
    b,Vo,npre,npost,ets=order_prepost(*E[lab])
    lvec=np.ones(npost)/npost
    olb,oub,out,bd=robust(b,Vo,npre,npost,lvec)
    print(f"[{lab}] post-avg original CI [{olb:+.4f},{oub:+.4f}]; breakdown Mbar={bd if bd else '>3'}")
    for mb,lb,ub,inc in out: print(f"   Mbar={mb}: [{lb:+.4f},{ub:+.4f}] {'INCL 0' if inc else 'excl 0'}")

print("\n=== (B) DIRECT honest-DiD on the BUFFER = No-PHI minus Has-PHI (CHE40) ===")
etN,attN,VN=E["No PHI"]; etH,attH,VH=E["Has PHI"]
assert np.array_equal(etN,etH)
attB=attN-attH; VB=VN+VH                      # disjoint samples -> independent -> covariances add
b,Vo,npre,npost,ets=order_prepost(etN,attB,VB)
post_ets=ets[ets>=0]
# k=0 target
lvec0=np.zeros(npost); lvec0[int(np.where(post_ets==0)[0][0])]=1.0
olb,oub,out,bd=robust(b,Vo,npre,npost,lvec0)
print(f"buffer at k=0: point={b[np.where(ets==0)[0][0]]:+.4f}; original CI [{olb:+.4f},{oub:+.4f}]; breakdown Mbar={bd if bd else '>3'}")
for mb,lb,ub,inc in out: print(f"   Mbar={mb}: [{lb:+.4f},{ub:+.4f}] {'INCL 0' if inc else 'excl 0'}")
# post-average buffer
lvecA=np.ones(npost)/npost
olbA,oubA,outA,bdA=robust(b,Vo,npre,npost,lvecA)
print(f"buffer post-average: original CI [{olbA:+.4f},{oubA:+.4f}]; breakdown Mbar={bdA if bdA else '>3'}")
for mb,lb,ub,inc in outA: print(f"   Mbar={mb}: [{lb:+.4f},{ub:+.4f}] {'INCL 0' if inc else 'excl 0'}")
pd.DataFrame([{"target":"buffer_k0","mbar":mb,"lb":lb,"ub":ub} for mb,lb,ub,_ in out]+
             [{"target":"buffer_postavg","mbar":mb,"lb":lb,"ub":ub} for mb,lb,ub,_ in outA]).to_csv(f"{TAB}/t19_buffer_honest.csv",index=False)
